Append all pickle files from EETCs_stat

In [1]:
import pickle
import os
from tqdm import tqdm

In [6]:
simulations = ['ERA5', 'UBB', 'UBD', 'UBE', 'UBF', 'UBG', 'UBH', 'UBI']
hist_future_map = {'UBG': 'UBD', 'UBH': 'UBE', 'UBI': 'UBF'}
original_selection = False
wetdays = False
future = True
metric = 'diff'

In [8]:
for sim in simulations:

    if sim in hist_future_map:
        hist_sim = hist_future_map[sim]
    
    EETC_dict_list = []
        
    if sim in ['UBB', 'ERA5']:
        end_year = 2023
    elif sim in ['UBG', 'UBH']:
        end_year = 2100
    elif sim == 'UBI':
        end_year = 2098
    else:
        end_year = 2014
    
    add_file = ''
    if not original_selection:
        if wetdays:
            add_file += '_wetdays'
        if future and sim in ['UBG', 'UBH', 'UBI']:
            add_file += '_future_percentile'
        add_file += f'_{metric}.pkl'
        
    output_file = f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/EETC/EETC_cum_{sim}_Quebec_1005hPa_1979-{end_year}_compound_8hrs_quantile_SSI{add_file}'

    if os.path.exists(output_file):
        os.remove(output_file)
        
    for iyear in tqdm(range(1979, end_year + 1)):
        if sim in hist_future_map:
            if iyear < 2015:
                simu = hist_future_map[sim]
            else:
                simu = sim
        else:
            simu = sim
        input_dir = f"/home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/EETC/{simu}/{iyear}"
        input_file = f'{input_dir}/EETC_cum_{simu}_Quebec_1005hPa_{iyear}_compound_8hrs_quantile_SSI{add_file}'
        try:
            with open(input_file, 'rb') as pickle_file:
                EETC_dict_iyear = pickle.load(pickle_file)
                EETC_dict_list.append(EETC_dict_iyear)
        except FileNotFoundError:
            continue

    EETC_dict = {k: v for d in EETC_dict_list for k, v in d.items()}

    with open(output_file, 'wb') as f:
        pickle.dump(EETC_dict, f)

100%|██████████| 120/120 [00:32<00:00,  3.73it/s]
